# Aegis: End-to-End Multi-Agent Demo

This notebook demonstrates:
- Mission → Multi-agent plan
- Agent execution
- Trace logging
- LLM-Judge scoring (mocked)
- AgentCreator generating new agents
- Re-running the mission with improved capabilities


In [1]:
!pip install rich

## 📌 Part 1 — Core Mocked Multi-Agent Framework

In [2]:
from typing import Dict, List, Any
from rich.pretty import pprint
import uuid, time

# ===== Trace system =====
TRACES = []

def trace_start(span_name, meta=None):
    span = {
        "id": str(uuid.uuid4()),
        "name": span_name,
        "start": time.time(),
        "meta": meta or {},
        "events": []
    }
    TRACES.append(span)
    return span

def trace_end(span, result):
    span["end"] = time.time()
    span["duration"] = span["end"] - span["start"]
    span["result"] = result
    span["events"].append({"event": "end", "t": time.time()})

# ===== Agent registry =====
AGENT_REGISTRY = {}

def register_agent(name, description, func):
    AGENT_REGISTRY[name] = {"name": name, "desc": description, "call": func}
    return AGENT_REGISTRY[name]

def call_agent(name, args, context):
    agent = AGENT_REGISTRY[name]
    span = trace_start(f"agent_call:{name}", {"args": args})
    result = agent["call"](args, context)
    trace_end(span, result)
    return result

# ===== Specialist Agents =====
def market_research_agent(args, ctx):
    q = args.get("query")
    return {"insights": f"Mock insights on '{q}' (competitors, keywords)", "confidence": 0.85}

def copy_agent(args, ctx):
    brief = args.get("brief")
    return {"copy": f"Landing page headline for '{brief}': Aegis helps you launch fast!"}

def webdev_agent(args, ctx):
    return {"artifact": {"url": "https://example.com/landing-demo"}}

register_agent("MarketResearchAgent", "Market research agent", market_research_agent)
register_agent("CopyAgent", "Writes marketing copy", copy_agent)
register_agent("WebDevAgent", "Builds landing pages", webdev_agent)

# ===== Judge =====
def llm_judge(mission, trace):
    names = [s["name"] for s in TRACES]
    score = 0
    if any("MarketResearchAgent" in n for n in names): score += 0.4
    if any("CopyAgent" in n for n in names): score += 0.3
    if any("WebDevAgent" in n for n in names): score += 0.3
    return {"score": round(min(score, 1.0), 2), "notes": "Mock judge evaluation"}

# ===== AgentCreator =====
def agent_creator_suggest(missing_cap):
    return {
        "name": f"{missing_cap}Agent",
        "desc": f"Automatically created agent for {missing_cap}",
        "example": {"input": "campaign", "output": "analytics"}
    }

def orchestrate_mission(mission_text, context):
    plan = [
        {"step": "market_research", "agent": "MarketResearchAgent", "args": {"query": mission_text}},
        {"step": "copy", "agent": "CopyAgent", "args": {"brief": mission_text}},
        {"step": "deploy", "agent": "WebDevAgent", "args": {"brief": mission_text}}
    ]

    results = {}
    for s in plan:
        results[s["step"]] = call_agent(s["agent"], s["args"], context)

    judge = llm_judge(mission_text, TRACES)
    return {"results": results, "judge": judge, "trace": TRACES}


## 📌 Part 2 — Run a mission

In [3]:
mission = "Launch a micro marketing campaign for Product X targeting students in Bangalore"
ctx = {}

out = orchestrate_mission(mission, ctx)
pprint(out)

## 📌 Part 3 — Self-Evolving AgentCreator

In [4]:
if out["judge"]["score"] < 0.9:
    missing = "Analytics"
    new_spec = agent_creator_suggest(missing)
    print("Generated new agent spec:")
    pprint(new_spec)

    def analytics_agent(args, ctx):
        return {"report": "Mock analytics report: campaign performing well"}

    register_agent(new_spec["name"], new_spec["desc"], analytics_agent)
    print("Registered:", new_spec["name"])

## 📌 Part 4 — Re-run mission with new agent included

In [5]:
def orchestrate_with_analytics(mission_text, context):
    plan = [
        {"step": "market_research", "agent": "MarketResearchAgent", "args": {"query": mission_text}},
        {"step": "copy", "agent": "CopyAgent", "args": {"brief": mission_text}},
        {"step": "deploy", "agent": "WebDevAgent", "args": {"brief": mission_text}},
        {"step": "analytics", "agent": "AnalyticsAgent", "args": {"campaign": "Product X"}}
    ]
    results = {}
    for s in plan:
        results[s["step"]] = call_agent(s["agent"], s["args"], context)
    judge = llm_judge(mission_text, TRACES)
    return {"results": results, "judge": judge}

out2 = orchestrate_with_analytics(mission, ctx)
pprint(out2)